# 🎨 Monet Style Transfer with CycleGAN
### Kaggle Competition: I'm Something of a Painter Myself

This notebook implements a CycleGAN to transform real photographs into Monet-style paintings.

In [ ]:
# --- 0. IMPORTS & CONFIGURATION ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np
import os
import shutil
import time

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

# ── Hyperparameters ──────────────────────────────────────────────────────────
IMAGE_SIZE   = [256, 256]
BATCH_SIZE   = 1          # CycleGAN papers use batch=1 with InstanceNorm
EPOCHS       = 25
LAMBDA_CYCLE = 10         # Cycle-consistency loss weight
LAMBDA_ID    = 5          # Identity loss weight (0.5 * lambda_cycle)
LEARNING_RATE = 2e-4
AUTOTUNE     = tf.data.AUTOTUNE  # Removed deprecated .experimental

In [ ]:
# --- 1. CUSTOM LAYER: InstanceNormalization ---
# Defined here for Kaggle compatibility (no tensorflow_addons required)

class InstanceNormalization(layers.Layer):
    """Instance Normalization Layer (Ulyanov et al., 2016).
    
    Normalizes across spatial dimensions (H, W) independently per sample
    and channel — preferred over BatchNorm for style-transfer tasks.
    """

    def __init__(self, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        n_channels = input_shape[-1]
        self.scale  = self.add_weight(
            name='scale', shape=(n_channels,),
            initializer=tf.random_normal_initializer(1.0, 0.02),
            trainable=True)
        self.offset = self.add_weight(
            name='offset', shape=(n_channels,),
            initializer='zeros', trainable=True)
        super().build(input_shape)

    def call(self, x):
        mean, variance = tf.nn.moments(x, axes=[1, 2], keepdims=True)
        inv = tf.math.rsqrt(variance + self.epsilon)
        return self.scale * (x - mean) * inv + self.offset

    def get_config(self):
        config = super().get_config()
        config.update({'epsilon': self.epsilon})
        return config

In [ ]:
# --- 2. DATA LOADING & AUGMENTATION ---

def decode_image(image):
    """Decode JPEG bytes → float32 tensor in [-1, 1] range."""
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1.0          # Normalize to [-1, 1]
    image = tf.image.resize(image, IMAGE_SIZE)  # FIX: explicit resize for robustness
    return image

def read_tfrecord(example):
    tfrecord_format = {"image": tf.io.FixedLenFeature([], tf.string)}
    example = tf.io.parse_single_example(example, tfrecord_format)
    return decode_image(example['image'])

def augment(image):
    """Random flips for data augmentation."""
    image = tf.image.random_flip_left_right(image)
    # Subtle random brightness jitter to improve generalization
    image = tf.clip_by_value(
        image + tf.random.normal(tf.shape(image), stddev=0.02), -1.0, 1.0)
    return image

def load_dataset(filenames, augment_data=False, shuffle=True):
    """Load a TFRecord dataset with optional augmentation and shuffling."""
    dataset = tf.data.TFRecordDataset(filenames)
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000, reshuffle_each_iteration=True)
    dataset = dataset.map(read_tfrecord, num_parallel_calls=AUTOTUNE)
    if augment_data:
        dataset = dataset.map(augment, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)   # FIX: prefetch for GPU pipeline efficiency
    return dataset

# ── Load data ────────────────────────────────────────────────────────────────
MONET_FILENAMES = tf.io.gfile.glob('/kaggle/input/gan-getting-started/monet_tfrec/*.tfrec')
PHOTO_FILENAMES = tf.io.gfile.glob('/kaggle/input/gan-getting-started/photo_tfrec/*.tfrec')

print(f'Monet TFRecords : {len(MONET_FILENAMES)}')
print(f'Photo TFRecords : {len(PHOTO_FILENAMES)}')

monet_ds = load_dataset(MONET_FILENAMES, augment_data=True)
photo_ds = load_dataset(PHOTO_FILENAMES, augment_data=True)

# Dataset for generation only (no shuffle / augment)
photo_ds_gen = load_dataset(PHOTO_FILENAMES, augment_data=False, shuffle=False)

In [ ]:
# --- 3. MODEL ARCHITECTURE ---

def resnet_block(n_filters, input_layer):
    """One ResNet residual block with reflection-style padding."""
    init = tf.random_normal_initializer(0.0, 0.02)
    g = layers.Conv2D(n_filters, (3, 3), padding='same', kernel_initializer=init,
                      use_bias=False)(input_layer)   # FIX: use_bias=False with IN
    g = InstanceNormalization()(g)
    g = layers.Activation('relu')(g)
    g = layers.Conv2D(n_filters, (3, 3), padding='same', kernel_initializer=init,
                      use_bias=False)(g)
    g = InstanceNormalization()(g)
    g = layers.Add()([g, input_layer])               # Skip connection
    return g


def build_generator(n_resnet=9):
    """U-Net style generator with residual bottleneck (Johnson et al., 2016).
    
    Args:
        n_resnet: Number of ResNet blocks in the bottleneck (9 for 256x256).
    """
    init   = tf.random_normal_initializer(0.0, 0.02)
    inputs = layers.Input(shape=[*IMAGE_SIZE, 3])

    # ── Encoder ──────────────────────────────────────────────────────────────
    x = layers.Conv2D(64, (7, 7), padding='same', kernel_initializer=init,
                      use_bias=False)(inputs)
    x = InstanceNormalization()(x)
    x = layers.Activation('relu')(x)

    for n_filters in [128, 256]:
        x = layers.Conv2D(n_filters, (3, 3), strides=(2, 2), padding='same',
                          kernel_initializer=init, use_bias=False)(x)
        x = InstanceNormalization()(x)
        x = layers.Activation('relu')(x)

    # ── Bottleneck (ResNet blocks) ────────────────────────────────────────────
    for _ in range(n_resnet):
        x = resnet_block(256, x)

    # ── Decoder ──────────────────────────────────────────────────────────────
    for n_filters in [128, 64]:
        x = layers.Conv2DTranspose(n_filters, (3, 3), strides=(2, 2), padding='same',
                                   kernel_initializer=init, use_bias=False)(x)
        x = InstanceNormalization()(x)
        x = layers.Activation('relu')(x)

    outputs = layers.Conv2D(3, (7, 7), padding='same', activation='tanh',
                            kernel_initializer=init)(inputs if False else x)
    return keras.Model(inputs, outputs, name='generator')


def build_discriminator():
    """PatchGAN discriminator (70x70 receptive field)."""
    init   = tf.random_normal_initializer(0.0, 0.02)
    inputs = layers.Input(shape=[*IMAGE_SIZE, 3])

    x = layers.Conv2D(64, (4, 4), strides=(2, 2), padding='same',
                      kernel_initializer=init)(inputs)
    x = layers.LeakyReLU(0.2)(x)

    for n_filters in [128, 256]:
        x = layers.Conv2D(n_filters, (4, 4), strides=(2, 2), padding='same',
                          kernel_initializer=init, use_bias=False)(x)
        x = InstanceNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)

    # FIX: Add a 512-filter stride-1 layer (standard PatchGAN design)
    x = layers.Conv2D(512, (4, 4), strides=(1, 1), padding='same',
                      kernel_initializer=init, use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    outputs = layers.Conv2D(1, (4, 4), padding='same', kernel_initializer=init)(x)
    return keras.Model(inputs, outputs, name='discriminator')


# ── Instantiate models ───────────────────────────────────────────────────────
monet_generator      = build_generator()
photo_generator      = build_generator()
monet_discriminator  = build_discriminator()
photo_discriminator  = build_discriminator()

print('Generator params     :', monet_generator.count_params():,)
print('Discriminator params :', monet_discriminator.count_params():,)

In [ ]:
# --- 4. LOSS FUNCTIONS ---

loss_obj = keras.losses.BinaryCrossentropy(from_logits=True, reduction='sum_over_batch_size')

def generator_loss(fake):
    """Generator wants discriminator to classify fakes as real."""
    return loss_obj(tf.ones_like(fake), fake)

def discriminator_loss(real, fake):
    """Discriminator wants to classify real as real and fake as fake."""
    real_loss = loss_obj(tf.ones_like(real),  real)
    fake_loss = loss_obj(tf.zeros_like(fake), fake)
    return (real_loss + fake_loss) * 0.5

def calc_cycle_loss(real, cycled, lam=LAMBDA_CYCLE):
    """L1 cycle-consistency loss."""
    return lam * tf.reduce_mean(tf.abs(real - cycled))

def identity_loss(real, same, lam=LAMBDA_CYCLE):
    """Identity loss — penalizes unnecessary changes to already-correct domain."""
    return lam * 0.5 * tf.reduce_mean(tf.abs(real - same))

In [ ]:
# --- 5. CYCLEGAN TRAINING CLASS ---

class CycleGan(keras.Model):
    """CycleGAN model (Zhu et al., 2017).
    
    Learns unpaired image-to-image translation via cycle-consistency.
    Here: Photo → Monet painting, Monet → Photo.
    """

    def __init__(self, m_gen, p_gen, m_disc, p_disc, lambda_cycle=LAMBDA_CYCLE):
        super().__init__()
        self.m_gen  = m_gen
        self.p_gen  = p_gen
        self.m_disc = m_disc
        self.p_disc = p_disc
        self.lambda_cycle = lambda_cycle

    def compile(self, m_gen_optimizer, p_gen_optimizer,
                m_disc_optimizer, p_disc_optimizer,
                gen_loss_fn, disc_loss_fn, cycle_loss_fn, identity_loss_fn):
        super().compile()
        self.m_gen_optimizer  = m_gen_optimizer
        self.p_gen_optimizer  = p_gen_optimizer
        self.m_disc_optimizer = m_disc_optimizer
        self.p_disc_optimizer = p_disc_optimizer
        self.gen_loss_fn      = gen_loss_fn
        self.disc_loss_fn     = disc_loss_fn
        self.cycle_loss_fn    = cycle_loss_fn
        self.identity_loss_fn = identity_loss_fn

    @tf.function
    def train_step(self, batch_data):
        real_monet, real_photo = batch_data

        with tf.GradientTape(persistent=True) as tape:
            # ── Forward passes ───────────────────────────────────────────────
            fake_monet   = self.m_gen(real_photo,  training=True)  # Photo  → Monet
            cycled_photo = self.p_gen(fake_monet,  training=True)  # Monet  → Photo (cycle)
            fake_photo   = self.p_gen(real_monet,  training=True)  # Monet  → Photo
            cycled_monet = self.m_gen(fake_photo,  training=True)  # Photo  → Monet (cycle)
            same_monet   = self.m_gen(real_monet,  training=True)  # Identity: Monet → Monet
            same_photo   = self.p_gen(real_photo,  training=True)  # Identity: Photo → Photo

            # ── Discriminator outputs ─────────────────────────────────────────
            disc_real_monet = self.m_disc(real_monet,  training=True)
            disc_fake_monet = self.m_disc(fake_monet,  training=True)
            disc_real_photo = self.p_disc(real_photo,  training=True)
            disc_fake_photo = self.p_disc(fake_photo,  training=True)

            # ── Generator losses ──────────────────────────────────────────────
            m_gen_loss = self.gen_loss_fn(disc_fake_monet)
            p_gen_loss = self.gen_loss_fn(disc_fake_photo)

            total_cycle = (
                self.cycle_loss_fn(real_monet, cycled_monet, self.lambda_cycle) +
                self.cycle_loss_fn(real_photo, cycled_photo, self.lambda_cycle)
            )

            total_m_gen_loss = (m_gen_loss + total_cycle +
                                self.identity_loss_fn(real_monet, same_monet, self.lambda_cycle))
            total_p_gen_loss = (p_gen_loss + total_cycle +
                                self.identity_loss_fn(real_photo, same_photo, self.lambda_cycle))

            # ── Discriminator losses ──────────────────────────────────────────
            m_disc_loss = self.disc_loss_fn(disc_real_monet, disc_fake_monet)
            p_disc_loss = self.disc_loss_fn(disc_real_photo, disc_fake_photo)

        # ── Apply gradients ───────────────────────────────────────────────────
        self.m_gen_optimizer.apply_gradients(zip(
            tape.gradient(total_m_gen_loss, self.m_gen.trainable_variables),
            self.m_gen.trainable_variables))
        self.p_gen_optimizer.apply_gradients(zip(
            tape.gradient(total_p_gen_loss, self.p_gen.trainable_variables),
            self.p_gen.trainable_variables))
        self.m_disc_optimizer.apply_gradients(zip(
            tape.gradient(m_disc_loss, self.m_disc.trainable_variables),
            self.m_disc.trainable_variables))
        self.p_disc_optimizer.apply_gradients(zip(
            tape.gradient(p_disc_loss, self.p_disc.trainable_variables),
            self.p_disc.trainable_variables))

        del tape   # Free persistent tape memory

        return {
            'm_gen_loss':  total_m_gen_loss,
            'p_gen_loss':  total_p_gen_loss,
            'm_disc_loss': m_disc_loss,
            'p_disc_loss': p_disc_loss,
        }

In [ ]:
# --- 6. COMPILE & TRAIN ---

# FIX: Use linear decay for learning rate after epoch 10 (as per original paper)
def lr_schedule(epoch, lr):
    if epoch < EPOCHS // 2:
        return lr
    return lr * (1.0 - (epoch - EPOCHS // 2) / (EPOCHS // 2 + 1))

lr_callback = keras.callbacks.LearningRateScheduler(lr_schedule, verbose=0)

model = CycleGan(monet_generator, photo_generator,
                 monet_discriminator, photo_discriminator)

model.compile(
    m_gen_optimizer  = keras.optimizers.Adam(LEARNING_RATE, beta_1=0.5),
    p_gen_optimizer  = keras.optimizers.Adam(LEARNING_RATE, beta_1=0.5),
    m_disc_optimizer = keras.optimizers.Adam(LEARNING_RATE, beta_1=0.5),
    p_disc_optimizer = keras.optimizers.Adam(LEARNING_RATE, beta_1=0.5),
    gen_loss_fn      = generator_loss,
    disc_loss_fn     = discriminator_loss,
    cycle_loss_fn    = calc_cycle_loss,
    identity_loss_fn = identity_loss,
)

start_time = time.time()

history = model.fit(
    tf.data.Dataset.zip((monet_ds, photo_ds)),
    epochs=EPOCHS,
    callbacks=[lr_callback],
)

print(f'Training time: {(time.time() - start_time) / 60:.1f} min')

In [ ]:
# --- 7. VISUALIZE RESULTS ---

def visualize_samples(photo_ds, n=5):
    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    axes[0, 0].set_title('Original Photo', fontsize=14, fontweight='bold')
    axes[0, 1].set_title('Monet Style',    fontsize=14, fontweight='bold')

    for i, img_batch in enumerate(photo_ds.take(n)):
        prediction = monet_generator(img_batch, training=False)[0]

        # Denormalize [-1,1] → [0,1]
        orig = (img_batch[0].numpy() * 0.5 + 0.5).clip(0, 1)
        pred = (prediction.numpy()   * 0.5 + 0.5).clip(0, 1)

        axes[i, 0].imshow(orig)
        axes[i, 0].axis('off')
        axes[i, 1].imshow(pred)
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.savefig('monet_samples.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_samples(photo_ds_gen)

In [ ]:
# --- 8. SAVE MODEL ---

# Save in modern .keras format (FIX: .h5 deprecated for subclassed layers)
monet_generator.save('monet_generator.keras')
print('Model saved as monet_generator.keras')

# Also export as SavedModel for TF Serving / ONNX compatibility
monet_generator.export('monet_generator_savedmodel')
print('SavedModel exported to monet_generator_savedmodel/')

In [ ]:
# --- 9. KAGGLE SUBMISSION ---

output_dir = '../images'
os.makedirs(output_dir, exist_ok=True)

print('Generating images...')
for i, img_batch in enumerate(photo_ds_gen):
    # Run inference
    prediction = monet_generator(img_batch, training=False)[0].numpy()
    # Denormalize: [-1,1] → [0,255]
    prediction = ((prediction * 127.5) + 127.5).clip(0, 255).astype(np.uint8)
    plt.imsave(f'{output_dir}/{i+1}.jpg', prediction)

    if (i + 1) % 500 == 0:
        print(f'  Generated {i+1} images...')

print(f'Done! Total images: {i+1}')

# Create submission zip
shutil.make_archive('/kaggle/working/images', 'zip', output_dir)
print('Submission archive: /kaggle/working/images.zip')